# YOLO-World BDD10K Training Pipeline

Notebook ini adalah workflow interaktif untuk training YOLO-World BDD10K. Logic utama tetap berada di `scripts/yoloworld_bdd10k_pipeline.py` agar sinkron dengan runner terminal/nohup.

## Project overview

Tujuan: fine-tune pretrained YOLO-World dari Ultralytics pada BDD10K 10 kelas, dengan flag untuk menentukan known classes yang dipelajari dan unknown classes yang disembunyikan dari supervised labels.

## Install dependencies

Jalankan di terminal jika environment belum siap:

```bash
pip install -r requirements.txt
```

In [ ]:
# Import libraries
from pathlib import Path
import os
import sys
import json
import yaml

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from yoloworld_bdd10k_pipeline import BDD10K_NAMES, build_parser, run_pipeline, load_yaml

PROJECT_ROOT

In [ ]:
# Define paths
DATA_YAML = PROJECT_ROOT / 'data/bdd10k/bdd10k.yaml'
OUTPUT_DIR = PROJECT_ROOT / 'runs/yoloworld_bdd10k'
EXPERIMENT_NAME = 'yoloworld_bdd10k_finetune'
MODEL = 'yolov8s-world.pt'
KNOWN_CLASSES = 'car,bus,truck'
UNKNOWN_PROMPTS = 'pedestrian,rider,train,motorcycle,bicycle,traffic light,traffic sign'

DATA_YAML, OUTPUT_DIR

In [ ]:
# Define BDD10K class names
BDD10K_NAMES

In [ ]:
# Check dataset structure
config = load_yaml(DATA_YAML)
dataset_root = Path(config.get('path', DATA_YAML.parent))
if not dataset_root.is_absolute():
    dataset_root = PROJECT_ROOT / dataset_root

for split in ['train', 'val', 'test']:
    image_dir = dataset_root / config.get(split, f'images/{split}')
    label_dir = Path(str(image_dir).replace('/images/', '/labels/'))
    print(split, 'images:', image_dir, image_dir.exists(), 'labels:', label_dir, label_dir.exists())

## Optional conversion BDD10K JSON to YOLO format

Gunakan script berikut dari terminal jika label YOLO belum tersedia:

```bash
python scripts/convert_bdd10k_to_yolo.py \
  --input-json data/bdd10k/labels/bdd100k_labels_images_train.json \
  --image-dir data/bdd10k/images/train \
  --output-label-dir data/bdd10k/labels/train
```

In [ ]:
# Create data YAML if needed
DATA_YAML.parent.mkdir(parents=True, exist_ok=True)
if not DATA_YAML.exists():
    DATA_YAML.write_text(yaml.safe_dump({
        'path': 'data/bdd10k',
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'names': BDD10K_NAMES,
    }, sort_keys=False), encoding='utf-8')
print(DATA_YAML.read_text(encoding='utf-8'))

In [ ]:
# Load YOLO-World pretrained model from Ultralytics
# This cell only verifies that the import path is available.
from ultralytics import YOLOWorld
print(YOLOWorld)

In [ ]:
# Train model
# Default notebook execution runs with the same defaults as the CLI runner.
RUN_TRAINING = True

parser = build_parser()
args = parser.parse_args([])

if RUN_TRAINING:
    train_summary = run_pipeline(args)
    train_summary
else:
    print('Training skipped. Set RUN_TRAINING = True to start.')
    print(vars(args))

In [ ]:
# Evaluate model
RUN_EVAL = False
if RUN_EVAL:
    eval_args = parser.parse_args([
        '--data-yaml', str(DATA_YAML),
        '--model', str(OUTPUT_DIR / EXPERIMENT_NAME / 'weights/best.pt'),
        '--output-dir', str(OUTPUT_DIR),
        '--experiment-name', EXPERIMENT_NAME,
        '--eval-only',
        '--imgsz', '640',
        '--batch-size', '8',
    ])
    eval_summary = run_pipeline(eval_args)
    eval_summary

In [ ]:
# Run inference on sample images
RUN_PREDICT = False
SOURCE = PROJECT_ROOT / 'data/bdd10k/images/val'
if RUN_PREDICT:
    pred_args = parser.parse_args([
        '--model', str(OUTPUT_DIR / EXPERIMENT_NAME / 'weights/best.pt'),
        '--output-dir', str(OUTPUT_DIR),
        '--experiment-name', EXPERIMENT_NAME,
        '--unknown-prompts', UNKNOWN_PROMPTS,
        '--predict-only',
        '--source', str(SOURCE),
        '--conf-thres', '0.25',
        '--iou-thres', '0.7',
    ])
    pred_summary = run_pipeline(pred_args)
    pred_summary

In [ ]:
# Visualize predictions
import matplotlib.pyplot as plt
import cv2

pred_dir = OUTPUT_DIR / EXPERIMENT_NAME / 'predictions'
pred_images = sorted([p for p in pred_dir.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]) if pred_dir.exists() else []
for path in pred_images[:4]:
    image = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 5))
    plt.imshow(image)
    plt.axis('off')
    plt.title(path.name)
    plt.show()

In [ ]:
# Save experiment summary
summary_path = OUTPUT_DIR / EXPERIMENT_NAME / 'notebook_summary.json'
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary = {
    'data_yaml': str(DATA_YAML),
    'model': MODEL,
    'known_classes': KNOWN_CLASSES,
    'unknown_prompts': UNKNOWN_PROMPTS,
    'output_dir': str(OUTPUT_DIR),
    'experiment_name': EXPERIMENT_NAME,
}
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
summary